In [ ]:
# Google Colab only
!pip install torchmetrics

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.metrics import f1_score

import torch
import torch.nn.functional as F

from torch import optim
from torch import nn
from torch.utils.data import random_split, DataLoader
from tqdm import tqdm

import torchvision

import torchvision.datasets as datasets
import torchvision.transforms as transforms
import torchvision.models as models

import torchmetrics
from itertools import product

In [ ]:
# Google Colab only
from google.colab import drive
drive.mount('/content/drive/')

Drive already mounted at /content/drive/; to attempt to forcibly remount, call drive.mount("/content/drive/", force_remount=True).


In [ ]:
# !ls '/content/drive/MyDrive/Colab Notebooks'

In [ ]:
# Obtain sampled datasets of benign and malignant tumor images
# Run 'sampling.ipynb' first to generate the data in 'samples/'

# --- your paths ---
# Jupyter path
# path = Path().resolve()

# Google Colab path
path = Path().resolve() / "drive" / "MyDrive"

samples = path / "samples"

plot_dir = path / "plots"
os.makedirs(plot_dir, exist_ok=True)

In [ ]:
# Device uses either CUDA or CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Use Cross Entropy Loss function as criterion
criterion = nn.CrossEntropyLoss()

print(torch.cuda.is_available())

True


In [ ]:
# Set different experiments in batch sizes and learning rates
batch_sizes = [16, 32]
learning_rates = [0.1, 0.01, 0.001]
all_results = {"resnext50": [], "custom_cnn": []}

# Compress images, convert to Tensor, and normalize to create transform function
transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Retrieve the sampled dataset from the directory. ImageFolder assigns classes to images (benign = 0; malignant = 1)
dataset = datasets.ImageFolder(root=samples, transform=transform)

# Define split sizes (70% training, 15% validation, 15% test)
total = len(dataset)
training_size = int(0.7 * total)
val_size = int(0.15 * total)
test_size = total - training_size - val_size # avoid rounding issues

# Split the dataset into training set, validation set, and test set.
train_set, val_set, test_set = torch.utils.data.random_split(dataset, [training_size, val_size, test_size])

In [ ]:
# Function to train model with validation
def train_model(model, train_loader, val_loader, optimizer, epochs):
    train_losses, val_losses = [], []
    train_accs, val_accs = [], []

    for epoch in range(epochs):
        # --- Training ---
        model.train()
        train_loss, train_acc = 0, 0

        # Train model on training set
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device) # images and labels are either sent to CUDA or CPU

            # Track optimizer's gradient descent
            optimizer.zero_grad() # zero out gradients from previous batch
            outputs = model(images) # forward propagation
            loss = criterion(outputs, labels) # use loss function to calculate loss
            loss.backward() # backward propagation to calculate gradients
            optimizer.step() # use gradients to update weights

            # Calculate the loss function to check the prediction of breast cancer images on training set
            train_loss += loss.item()
            # Accumulate all the correct predictions across all batches
            train_acc += (outputs.argmax(1) == labels).sum().item()

        # --- Validation ---
        model.eval()
        val_loss, val_acc = 0, 0

        # To save memory, avoid tracking gradient descent on the validation set
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device) # images and labels are either sent to CUDA or CPU
                outputs = model(images)
                loss = criterion(outputs, labels)

                # Calculate the loss function to check the prediction of breast cancer images on validation set
                val_loss += loss.item()
                # Accumulate all the correct predictions across all batches
                val_acc += (outputs.argmax(1) == labels).sum().item()

        # --- Statistics ---
        print(f"Epoch {epoch+1}/{epochs}")
        print(f"  Train Loss: {train_loss/len(train_loader):.4f} | Train Acc: {train_acc/len(train_loader.dataset):.4f}")
        print(f"  Val Loss:   {val_loss/len(val_loader):.4f}   | Val Acc: {val_acc/len(val_loader.dataset):.4f}")

        # Record accuracies and losses
        train_losses.append(train_loss / len(train_loader))
        val_losses.append(val_loss / len(val_loader))
        train_accs.append(train_acc / len(train_loader.dataset))
        val_accs.append(val_acc / len(val_loader.dataset))

    return train_losses, val_losses, train_accs, val_accs



In [ ]:
def evaluate_model(model, test_loader):
    model.eval()
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            preds = outputs.argmax(1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    test_acc = sum(p == l for p, l in zip(all_preds, all_labels)) / len(all_labels)
    f1 = f1_score(all_labels, all_preds, average='binary')

    print(f"  Test Accuracy: {test_acc:.4f}")
    print(f"  F1-Score:      {f1:.4f}")

    return test_acc, f1

In [ ]:
# Function to save plots in plot_dir
def save_plot(train_losses, val_losses, train_accs, val_accs, model_name, batch_size, lr):
    # Create the x-axis starting from 1 instead of 0
    epochs = range(1, len(train_losses) + 1)

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    # --- Loss Plot ---
    axes[0].plot(epochs, train_losses, label="Train Loss")
    axes[0].plot(epochs, val_losses, label="Val Loss")
    axes[0].set_title(f"Loss | Batch: {batch_size} | LR: {lr}")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Loss")
    axes[0].set_xticks(epochs)
    axes[0].legend()

    # --- Accuracy Plot ---
    axes[1].plot(epochs, train_accs, label="Train Acc")
    axes[1].plot(epochs, val_accs, label="Val Acc")
    axes[1].set_title(f"Accuracy | Batch: {batch_size} | LR: {lr}")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Accuracy")
    axes[1].set_xticks(epochs)
    axes[1].legend()

    plt.tight_layout()
    filename = f"{model_name}_batch{batch_size}_lr{lr}.png"
    plt.savefig(os.path.join(plot_dir, filename))
    plt.close()
    print(f"Plot saved: {filename}")

In [ ]:
print("=== ResNeXt-50 Grid Search ===")
for batch_size, lr in product(batch_sizes, learning_rates):
    print(f"\n--- Batch Size: {batch_size} | LR: {lr} ---")

    train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True, num_workers=4, pin_memory=True)
    val_loader = DataLoader(val_set, batch_size=batch_size, shuffle=False, num_workers=4, pin_memory=True)

    # Initialize ResNeXt-50 model
    resnext_model = models.resnext50_32x4d(weights=models.ResNeXt50_32X4D_Weights.DEFAULT)
    in_features = resnext_model.fc.in_features
    resnext_model.fc = nn.Sequential(
        nn.Linear(in_features, 256),
        nn.ReLU(),
        nn.Dropout(0.25),
        nn.Linear(256, 2)
    )
    resnext_model = resnext_model.to(device)
    optimizer = torch.optim.Adam(resnext_model.fc.parameters(), lr=lr)

    train_losses, val_losses, train_accs, val_accs = train_model(
        resnext_model, train_loader, val_loader, optimizer, epochs=10
    )

    all_results['resnext50'].append({
        'batch_size': batch_size,
        'lr': lr,
        'final_val_loss': val_losses[-1],
        'final_val_acc': val_accs[-1]
    })

    save_plot(train_losses, val_losses, train_accs, val_accs, 'resnext50', batch_size, lr)

    del resnext_model
    torch.cuda.empty_cache()

=== ResNeXt-50 Grid Search ===

--- Batch Size: 16 | LR: 0.1 ---
Epoch 1/10
  Train Loss: 5.2438 | Train Acc: 0.8571
  Val Loss:   0.1554   | Val Acc: 0.9444
Epoch 2/10
  Train Loss: 0.4005 | Train Acc: 0.8986
  Val Loss:   0.3787   | Val Acc: 0.8978
Epoch 3/10
  Train Loss: 0.3502 | Train Acc: 0.8995
  Val Loss:   0.3206   | Val Acc: 0.9178
Epoch 4/10
  Train Loss: 0.3286 | Train Acc: 0.8652
  Val Loss:   0.2868   | Val Acc: 0.8267
Epoch 5/10
  Train Loss: 0.3859 | Train Acc: 0.7919
  Val Loss:   0.3182   | Val Acc: 0.8711
Epoch 6/10
  Train Loss: 0.4397 | Train Acc: 0.7729
  Val Loss:   0.2003   | Val Acc: 0.9000
Epoch 7/10
  Train Loss: 0.5301 | Train Acc: 0.7519
  Val Loss:   0.5072   | Val Acc: 0.7156
Epoch 8/10
  Train Loss: 0.4886 | Train Acc: 0.7019
  Val Loss:   0.2168   | Val Acc: 0.9311
Epoch 9/10
  Train Loss: 0.3675 | Train Acc: 0.8014
  Val Loss:   0.2338   | Val Acc: 0.7978
Epoch 10/10
  Train Loss: 0.3375 | Train Acc: 0.8214
  Val Loss:   0.2495   | Val Acc: 0.9000
Plot

In [ ]:
# Our custom Convolutional Neural Network model
class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()

        # On each layer, apply 2D convolution on the image, starting with 3 input channels (RGB colors)
        # and 32 output channels, which gradually increases by a multiplication of 2.

        # --- Layer 1: Capture low-level patterns such as edges and textures ---
        self.layer1 = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)
        )

        # --- Layer 2: Capture middle-level patterns such as contours and shapes ---
        self.layer2 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)
        )

        # --- Deep layers: Capture high-level patterns such as bumps and dips ---
        self.layer3 = nn.Sequential(
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)
        )

        self.layer4 =  nn.Sequential(
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)
        )

        # --- Classification head ---
        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256 * 14 * 14, 256), # 256 - number of features; 14 * 14 - spatial size of feature map after 4 max pools
            nn.ReLU(),
            nn.Dropout(0.25),               # Drop 25% of the dataset to prevent overfitting
            nn.Linear(256, 2)               # Two class labels: benign or malignant
        )

    def forward(self, x):
      x = self.layer1(x)
      x = self.layer2(x)
      x = self.layer3(x)
      x = self.layer4(x)
      x = self.fc(x)
      return x

In [ ]:
print("=== Custom CNN Grid Search ===")
for batch_size, lr in product(batch_sizes, learning_rates):
    print(f"\n--- Batch Size: {batch_size} | LR: {lr} ---")

    train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True, num_workers=8, pin_memory=True)
    val_loader = DataLoader(val_set, batch_size=batch_size, shuffle=False, num_workers=8, pin_memory=True)

    cnn_model = CNN().to(device)
    optimizer = torch.optim.Adam(cnn_model.parameters(), lr=lr)

    train_losses, val_losses, train_accs, val_accs = train_model(
        cnn_model, train_loader, val_loader, optimizer, epochs=10
    )

    all_results["custom_cnn"].append({
        "batch_size": batch_size,
        "lr": lr,
        "final_val_loss": val_losses[-1],
        "final_val_acc": val_accs[-1]
    })

    save_plot(train_losses, val_losses, train_accs, val_accs, "custom_cnn", batch_size, lr)

    del cnn_model
    torch.cuda.empty_cache()

=== Custom CNN Grid Search ===

--- Batch Size: 16 | LR: 0.1 ---
Epoch 1/10
  Train Loss: 388197.3243 | Train Acc: 0.4986
  Val Loss:   0.7447   | Val Acc: 0.5244
Epoch 2/10
  Train Loss: 0.7056 | Train Acc: 0.4924
  Val Loss:   0.6931   | Val Acc: 0.5244
Epoch 3/10
  Train Loss: 0.6985 | Train Acc: 0.5024
  Val Loss:   0.6989   | Val Acc: 0.4756
Epoch 4/10
  Train Loss: 0.6971 | Train Acc: 0.5014
  Val Loss:   0.6953   | Val Acc: 0.4756
Epoch 5/10
  Train Loss: 0.6981 | Train Acc: 0.4986
  Val Loss:   0.6967   | Val Acc: 0.4756
Epoch 6/10
  Train Loss: 0.6989 | Train Acc: 0.4967
  Val Loss:   0.6971   | Val Acc: 0.4756
Epoch 7/10
  Train Loss: 0.7022 | Train Acc: 0.4814
  Val Loss:   0.6957   | Val Acc: 0.5244
Epoch 8/10
  Train Loss: 0.6964 | Train Acc: 0.4986
  Val Loss:   0.6958   | Val Acc: 0.5244
Epoch 9/10
  Train Loss: 0.6985 | Train Acc: 0.5052
  Val Loss:   0.6943   | Val Acc: 0.4756
Epoch 10/10
  Train Loss: 0.6978 | Train Acc: 0.5014
  Val Loss:   0.6950   | Val Acc: 0.5244

In [ ]:
# =========================================================
# 1. FINAL EVALUATION: RESNEXT-50
# =========================================================

# Setup loaders for best ResNeXt config
train_loader = DataLoader(train_set, batch_size=32, shuffle=True, num_workers=4, pin_memory=True)
val_loader = DataLoader(val_set, batch_size=32, shuffle=False, num_workers=4, pin_memory=True)
test_loader = DataLoader(test_set, batch_size=32, shuffle=False, num_workers=4, pin_memory=True)

# Initialize and train ResNeXt
resnext_model = models.resnext50_32x4d(weights=models.ResNeXt50_32X4D_Weights.DEFAULT)
resnext_model.fc = nn.Sequential(
    nn.Linear(resnext_model.fc.in_features, 256),
    nn.ReLU(),
    nn.Dropout(0.5),
    nn.Linear(256, 2)
)
resnext_model = resnext_model.to(device)
optimizer = torch.optim.Adam(resnext_model.fc.parameters(), lr=0.001)

train_model(resnext_model, train_loader, val_loader, optimizer, epochs=5)

print("\n=== ResNeXt-50 Test Results ===")
resnext_test_acc, resnext_f1 = evaluate_model(resnext_model, test_loader)
print()

del resnext_model
torch.cuda.empty_cache()

Epoch 1/5
  Train Loss: 0.2423 | Train Acc: 0.8967
  Val Loss:   0.1271   | Val Acc: 0.9556
Epoch 2/5
  Train Loss: 0.1238 | Train Acc: 0.9533
  Val Loss:   0.1372   | Val Acc: 0.9489
Epoch 3/5
  Train Loss: 0.0931 | Train Acc: 0.9624
  Val Loss:   0.1086   | Val Acc: 0.9667
Epoch 4/5
  Train Loss: 0.0901 | Train Acc: 0.9676
  Val Loss:   0.1697   | Val Acc: 0.9422
Epoch 5/5
  Train Loss: 0.0751 | Train Acc: 0.9710
  Val Loss:   0.0965   | Val Acc: 0.9600

=== ResNeXt-50 Test Results ===
  Test Accuracy: 0.9600
  F1-Score:      0.9617



In [ ]:
# =========================================================
# 2. FINAL EVALUATION: CUSTOM CNN
# =========================================================

# Setup loaders for best Custom CNN config (in case it's different from ResNeXt)
train_loader_c = DataLoader(train_set, batch_size=32, shuffle=True, num_workers=8, pin_memory=True)
val_loader_c = DataLoader(val_set, batch_size=32, shuffle=False, num_workers=8, pin_memory=True)
test_loader_c = DataLoader(test_set, batch_size=32, shuffle=False, num_workers=8, pin_memory=True)

# Initialize and train Custom CNN
custom_model = CNN().to(device)
optimizer_c = torch.optim.Adam(custom_model.parameters(), lr=0.001)

train_model(custom_model, train_loader_c, val_loader_c, optimizer_c, epochs=6)

print("\n=== Custom CNN Test Results ===")
custom_test_acc, custom_f1 = evaluate_model(custom_model, test_loader_c)

del custom_model
torch.cuda.empty_cache()

Epoch 1/6
  Train Loss: 0.5862 | Train Acc: 0.7205
  Val Loss:   0.4256   | Val Acc: 0.7889
Epoch 2/6
  Train Loss: 0.4025 | Train Acc: 0.8219
  Val Loss:   0.4022   | Val Acc: 0.8222
Epoch 3/6
  Train Loss: 0.3594 | Train Acc: 0.8529
  Val Loss:   0.3210   | Val Acc: 0.8467
Epoch 4/6
  Train Loss: 0.3138 | Train Acc: 0.8738
  Val Loss:   0.2978   | Val Acc: 0.8556
Epoch 5/6
  Train Loss: 0.2557 | Train Acc: 0.8967
  Val Loss:   0.2894   | Val Acc: 0.8667
Epoch 6/6
  Train Loss: 0.2161 | Train Acc: 0.9133
  Val Loss:   0.3305   | Val Acc: 0.8867

=== Custom CNN Test Results ===
  Test Accuracy: 0.9067
  F1-Score:      0.9062
